# 01 — Load Tier 1 data

Read raw CSV/XLSX files. Print shapes and dtypes. Cache typed copies as parquet so downstream notebooks do not pay Excel-parse cost.

**Upstream:** raw files in `data/`

**Output:** parquets in `pipeline/artifacts/` (sales, cb, tpr, inv_snapshot, item_master, vendor, po, slprsn_key)

**Promotes to:** `src/load.py` once verified.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
DATA = ROOT / 'data'
ART  = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

SALES_CSV   = DATA / 'POP_SalesTransactionHistory.csv'
INV_XLSX    = DATA / 'POP_InventorySnapshot.xlsx'
ITEM_XLSX   = DATA / 'POP_ItemSpecMaster.xlsx'
VENDOR_XLSX = DATA / 'POP_VendorMaster.xlsx'
PO_XLSX     = DATA / 'POP_PurchaseOrderHistory.XLSX'
CB_XLSX     = DATA / 'POP_ChargeBack_Deductions_Penalties_Freight.xlsx'
SLPRSN_XLSX = DATA / 'SLPRSNID_SALESCHANNEL_KEY.xlsx'

for p in [SALES_CSV, INV_XLSX, ITEM_XLSX, VENDOR_XLSX, PO_XLSX, CB_XLSX, SLPRSN_XLSX]:
    assert p.exists(), f'missing: {p}'

## 2. Read raw files

One source-of-truth load per file. Downstream notebooks read the cached parquets in step 5.

In [2]:
sales = pd.read_csv(SALES_CSV, parse_dates=['DOCDATE'], low_memory=False)

inv = pd.concat([
    pd.read_excel(INV_XLSX, sheet_name='Site 1 - SF').assign(DC='SF'),
    pd.read_excel(INV_XLSX, sheet_name='Site 2 - NJ').assign(DC='NJ'),
    pd.read_excel(INV_XLSX, sheet_name='Site 3 - LA').assign(DC='LA'),
], ignore_index=True)

items   = pd.read_excel(ITEM_XLSX,   sheet_name='Item Spec Master')
vendors = pd.read_excel(VENDOR_XLSX, sheet_name='Supplier Master')
pos     = pd.read_excel(PO_XLSX,     sheet_name='PO Order History 2023-2025')
cb      = pd.read_excel(CB_XLSX,     sheet_name='Data - Deductions & Cause Code')
slprsn  = pd.read_excel(SLPRSN_XLSX)   # default first sheet; adjust if needed

for name, df in [('sales',sales),('inv',inv),('items',items),('vendors',vendors),('pos',pos),('cb',cb),('slprsn',slprsn)]:
    print(f'{name:10s} {df.shape}  cols[:6]={list(df.columns)[:6]}')

sales      (236818, 23)  cols[:6]=['LOCNCODE', 'SLPRSNID', 'CUSTNMBR', 'CITY', 'STATE', 'ZIPCODE']
inv        (219, 5)  cols[:6]=['Item Number', 'Description', 'Available', 'On Hand', 'DC']
items      (65, 15)  cols[:6]=['Item Number', 'Description', 'Case Pack', 'unit dimension (L*W*H) (in)', 'inner case dimension (L*W*H) (in)', 'master case dimension (L*W*H) (in)']
vendors    (73, 17)  cols[:6]=['Brand', 'Product Line', 'Category', 'Abbr. Legend (for WH)', 'Vendor ID', 'State']
pos        (5281, 16)  cols[:6]=['PO Number', 'PO Date', 'Required Date', 'Promised Ship Date', 'Receipt Date', 'POP Receipt Number']
cb         (18804, 13)  cols[:6]=['Location Code', 'Salesperson ID', 'Customer Number', 'City from Sales Transaction', 'State from Sales Transaction', 'SOP Type']
slprsn     (46, 3)  cols[:6]=['SLPRSNID', 'SALESCHANNEL', 'SALESCHANNEL_DESC']


## 3. Derive TPR subset of chargebacks

Chargebacks have many cause codes. The TPR / promo codes are what drive `is_promo`; split them out here so downstream notebooks can grab just `tpr.parquet`.

In [3]:
tpr_mask = cb['Cause Code Desc'].str.contains('TPR|promo|price reduction', case=False, na=False)
tpr = cb[tpr_mask].copy()

print(f'tpr rows: {len(tpr)} of {len(cb)} chargeback rows ({len(tpr)/len(cb):.1%})')
print()
print('Top cause codes in tpr subset:')
print(tpr['Cause Code'].value_counts().head(15))

tpr rows: 6868 of 18804 chargeback rows (36.5%)

Top cause codes in tpr subset:
Cause Code
CRED03      4113
CRED05      1645
CRED02       571
CRED04       259
CRED10-D     150
CRED-PRO     130
Name: count, dtype: int64


## 4. Validate

In [4]:
print('sales date range   :', sales['DOCDATE'].min(), '->', sales['DOCDATE'].max())
print('unique SKUs (sales):', sales['ITEMNMBR'].nunique())
print('unique SKUs (items):', items.iloc[:, 0].nunique())
print('inv rows per DC    :', inv['DC'].value_counts().to_dict())
print('tpr customers      :', tpr['Customer Number'].nunique() if 'Customer Number' in tpr.columns else 'col missing')
print()
print('sales columns with nulls (top 10):')
print(sales.isna().sum().sort_values(ascending=False).head(10))

sales date range   : 2023-01-03 00:00:00 -> 2026-04-13 00:00:00
unique SKUs (sales): 83
unique SKUs (items): 64
inv rows per DC    : {'SF': 73, 'NJ': 73, 'LA': 73}
tpr customers      : 88

sales columns with nulls (top 10):
ZIPCODE           4975
STATE             3876
CITY              3748
Product Type       490
Customer Type        4
SLPRSNID             4
LOCNCODE             0
XTNDPRCE_adj         0
UOM_Price            0
Margin_Pct_adj       0
dtype: int64


## 5. Save downstream artifact

In [5]:
to_cache = {
    'sales':         sales,
    'cb':            cb,
    'tpr':           tpr,
    'inv_snapshot':  inv,
    'item_master':   items,
    'vendor_master': vendors,
    'po':            pos,
    'slprsn_key':    slprsn,
}

for name, df in to_cache.items():
    path = ART / f'{name}.parquet'
    try:
        df.to_parquet(path)
        print(f'{name:14s} {df.shape} -> {path.name}')
    except Exception as e:
        alt = ART / f'{name}.pkl'
        df.to_pickle(alt)
        print(f'{name:14s} parquet failed ({type(e).__name__}); saved pickle -> {alt.name}')

sales          parquet failed (ImportError); saved pickle -> sales.pkl
cb             parquet failed (ImportError); saved pickle -> cb.pkl
tpr            parquet failed (ImportError); saved pickle -> tpr.pkl
inv_snapshot   parquet failed (ImportError); saved pickle -> inv_snapshot.pkl
item_master    parquet failed (ImportError); saved pickle -> item_master.pkl
vendor_master  parquet failed (ImportError); saved pickle -> vendor_master.pkl
po             parquet failed (ImportError); saved pickle -> po.pkl
slprsn_key     parquet failed (ImportError); saved pickle -> slprsn_key.pkl


## 6. Promote

Once loads and sanity checks look right, move the read logic into `src/load.py` as a single `load_all()` returning a dict of dataframes. Downstream dev notebooks can then:

```python
from src.load import load_all
data = load_all()
sales, cb, tpr = data['sales'], data['cb'], data['tpr']
```

Keep this notebook as the human-readable "how the data was parsed" reference.